# Week 10 Lesson Exercise: How a CNN Sees an Image

**Estimated time: 25 minutes**

In this guided exercise, you will explore the main ideas behind a **Convolutional Neural Network (CNN)** before training a full image classifier.

You will learn how:

- An image becomes numbers
- Pixels and channels are stored in tensors
- A convolution filter looks for patterns
- A feature map stores detected patterns
- Pooling reduces the size of feature maps

You do **not** need advanced mathematics. The goal is to understand the ideas visually and through simple code.


## 1. Images are numerical data

A computer does not see an image the way a person does.

A grayscale image can be represented as a 2-D grid of numbers.

For example:

```text
0   0   0   0   0
0 255 255 255   0
0 255   0 255   0
0 255 255 255   0
0   0   0   0   0
```

Each number represents a **pixel intensity**.

- `0` means dark
- `255` means bright

Color images usually contain three channels:

- Red
- Green
- Blue

This means an image can be stored as a **tensor** with dimensions such as:

`height × width × channels`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a simple 8 × 8 grayscale image.
# 0 means black and 1 means white.
image = np.array([
    [0,0,0,0,0,0,0,0],
    [0,0,0,1,1,0,0,0],
    [0,0,0,1,1,0,0,0],
    [0,0,0,1,1,0,0,0],
    [0,0,0,1,1,0,0,0],
    [0,0,0,1,1,0,0,0],
    [0,0,0,1,1,0,0,0],
    [0,0,0,0,0,0,0,0]
], dtype=float)

print("Image shape:", image.shape)

plt.imshow(image, cmap="gray")
plt.title("A Simple Grayscale Image")
plt.axis("off")
plt.show()


### Check Your Understanding

1. What does `image.shape` tell you?
2. Is this image grayscale or RGB?
3. How many channels does a grayscale image usually have?

Write your answers in your notebook or discuss them with your instructor.


## 2. What is a convolution filter?

A **convolution filter**, also called a **kernel**, is a small matrix that moves across an image.

Its purpose is to detect local patterns such as:

- Vertical edges
- Horizontal edges
- Curves
- Textures

A CNN learns useful filters automatically during training.

For learning purposes, we will manually use a simple vertical-edge filter.


In [ ]:
# A simple 3 × 3 vertical-edge detector.
vertical_filter = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
], dtype=float)

print(vertical_filter)


## 3. Apply the filter to the image

The filter slides across small regions of the image.

At each position:

1. Take a small part of the image
2. Multiply it by the filter
3. Add the values
4. Store the result

The collection of results is called a **feature map**.


In [ ]:
def apply_filter(image, kernel):
    # Determine the output size.
    image_h, image_w = image.shape
    kernel_h, kernel_w = kernel.shape

    output_h = image_h - kernel_h + 1
    output_w = image_w - kernel_w + 1

    # Create an empty feature map.
    feature_map = np.zeros((output_h, output_w))

    # Slide the filter across the image.
    for row in range(output_h):
        for col in range(output_w):
            # Select a small image region.
            region = image[
                row:row + kernel_h,
                col:col + kernel_w
            ]

            # Multiply corresponding values and add them together.
            feature_map[row, col] = np.sum(region * kernel)

    return feature_map


feature_map = apply_filter(image, vertical_filter)

print("Feature map shape:", feature_map.shape)
print(feature_map)


In [ ]:
plt.imshow(feature_map, cmap="gray")
plt.title("Feature Map from Vertical-Edge Filter")
plt.axis("off")
plt.show()


### What happened?

The filter responds strongly where it detects a change from dark to light or light to dark.

That is why CNNs are powerful for images:

> They learn filters that detect useful visual patterns.

Early CNN layers may detect simple patterns such as edges.

Deeper layers can combine those patterns into more complex features.


## 4. Student Experiment: Try a Different Filter

Now try a horizontal-edge detector.

Replace the values in the kernel below with:

```text
-1 -1 -1
 0  0  0
 1  1  1
```

Then run the code.

### Before running

Predict:

- Will the new feature map look the same?
- What kind of pattern should the filter detect?


In [ ]:
# STUDENT TASK:
# Replace this kernel with the horizontal-edge filter shown above.

horizontal_filter = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
], dtype=float)

horizontal_feature_map = apply_filter(image, horizontal_filter)

plt.imshow(horizontal_feature_map, cmap="gray")
plt.title("Feature Map from Your Filter")
plt.axis("off")
plt.show()


## 5. Pooling

Feature maps can still be large.

**Pooling** reduces their size while trying to keep important information.

A common method is **max pooling**.

For each small region, max pooling keeps only the largest value.

Example:

```text
1  5
3  2
```

becomes:

```text
5
```

This reduces computation and helps the network focus on strong features.


In [ ]:
def max_pool_2x2(feature_map):
    # Pool non-overlapping 2 × 2 regions.
    output_h = feature_map.shape[0] // 2
    output_w = feature_map.shape[1] // 2

    pooled = np.zeros((output_h, output_w))

    for row in range(output_h):
        for col in range(output_w):
            region = feature_map[
                row * 2:row * 2 + 2,
                col * 2:col * 2 + 2
            ]
            pooled[row, col] = np.max(region)

    return pooled


pooled_map = max_pool_2x2(feature_map)

print("Before pooling:", feature_map.shape)
print("After pooling:", pooled_map.shape)
print(pooled_map)


In [ ]:
plt.imshow(pooled_map, cmap="gray")
plt.title("Feature Map After 2 × 2 Max Pooling")
plt.axis("off")
plt.show()


## 6. Connect This to a Real CNN

A basic CNN often follows this pattern:

```text
Image
  ↓
Convolution
  ↓
Activation Function
  ↓
Pooling
  ↓
More Convolution / Pooling
  ↓
Classification Layers
  ↓
Prediction
```

The important idea is:

- **Convolution layers** detect patterns
- **Feature maps** store detected patterns
- **Pooling layers** reduce size
- **Classification layers** use the learned features to choose a class


## Quick Check

Answer in plain English:

1. What is a pixel?
2. What is an image channel?
3. What is a convolution filter?
4. What is a feature map?
5. What does max pooling do?
6. Why might CNNs work well for images?

### Key Takeaway

A CNN does not memorize an image as one giant list of numbers.

It learns **visual patterns** from small regions and combines those patterns to make predictions.
